# Lesson 19 Lab — ONNX Export, Graph Repair, and Shape Consistency

**Puzzle:** Can a pruned PyTorch model run correctly while its exported graph carries inconsistent channel metadata?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Physical pruning changes dimensions across weights, bias, normalization, reshape, concat, and post-processing nodes. ONNX export success only serializes the traced path; checker, shape inference, ONNX Runtime parity, and explicit dimension audits establish a deployable graph.


## 0. Predict before running

1. Predict the output channel dimension after the physical slice.
2. Explain what ONNX shape inference can and cannot establish.
3. Choose a parity tolerance for the independent runtime output.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

A small physically pruned multi-input model is exported in memory/on disk, checked with ONNX, passed through shape inference, executed with ONNX Runtime when available, and compared with CUDA/PyTorch output.

- Export, checker, inference, and runtime parity are distinct gates.
- Physical channel changes must reach initializers and consumer shapes.
- Dynamic symbols do not excuse inconsistent known dimensions.


## 2. Derive the mechanism

Static tensor shapes encode known dimensions while dynamic axes use symbolic parameters. Shape inference propagates what operator schemas can prove but cannot resolve every data-dependent reshape. `onnx.checker.check_model(..., full_check=True)` validates graph structure and types; ONNX Runtime supplies an independent execution path. After channel deletion, every initializer and consumer dimension must agree with the new graph contract.

### Mechanism at a glance

```mermaid
flowchart LR
  P["physically pruned model"] --> E["ONNX export"]
  E --> C["onnx.checker"]
  C --> S["shape inference"]
  S --> R["ONNX Runtime execution"]
  R --> V["shape + numerical comparison"]
  V -->|"fail"| L["repair index, bias,<br/>merge, or postprocess ledger"]
  L --> E
```

### Walk it step by step

1. **Build an index ledger.** For every removed channel, record the affected weight, bias, normalization, merge, and consumer dimensions.
2. **Export the structural candidate.** Use representative inputs and explicit dynamic-axis rules instead of treating export success as validation.
3. **Run graph checks in order.** Apply ONNX checker, shape inference, and a runtime execution with known inputs.
4. **Compare semantics.** Match output names, shapes, and numerical values with the framework candidate before accepting the graph.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 19
LESSON_TITLE = 'ONNX Export, Graph Repair, and Shape Consistency'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260827
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | PyTorch output and declared pruned shape ledger |
| Candidate | checked/inferred ONNX graph plus ONNX Runtime execution |
| Held constant | weights, retained indices, inputs, opset, dynamic-axis policy, dtype, and tolerance |
| Measurements | export status, checker status, inferred shapes, initializer dimensions, runtime status, and max error |
| Evidence | `native-backend` |

**Experiment:** Export a physically pruned CUDA model, run ONNX checker and shape inference, and compare ONNX Runtime output.


## 5. Read the experiment code

The notebook writes the ONNX model into the lesson artifact directory, invokes full checking, records inferred value shapes, and runs the same inputs through ONNX Runtime. Exceptions are caught into structured fields, but success requires every gate and numerical parity rather than export alone.

Do not execute until the code implements the frozen table above.


In [2]:
class PrunedMultiInput(nn.Module):
    def __init__(self): super().__init__(); self.proj=nn.Linear(12,7); self.gate=nn.Linear(5,7,bias=False); self.out=nn.Linear(7,3)
    def forward(self,x,context): return self.out(F.gelu(self.proj(x)+self.gate(context)))
model=PrunedMultiInput().to(DEVICE).eval(); x=torch.randn(4,12,device=DEVICE); context=torch.randn(4,5,device=DEVICE)
with torch.inference_mode(): ref=model(x,context).cpu().numpy()
onnx_available=importlib.util.find_spec("onnx") is not None; ort_available=importlib.util.find_spec("onnxruntime") is not None
artifact_dir=Path("artifacts"); artifact_dir.mkdir(exist_ok=True); onnx_path=artifact_dir/"pruned-multi-input.onnx"
exported=checker=False; inferred_ok=False; ort_ok=False; ort_error=float("nan"); inferred_shapes=[]; message=""
try:
    torch.onnx.export(model,(x,context),onnx_path,input_names=["x","context"],output_names=["logits"],dynamic_axes={"x":{0:"batch"},"context":{0:"batch"},"logits":{0:"batch"}},opset_version=17,dynamo=False)
    exported=True
    import onnx
    loaded=onnx.load(onnx_path); onnx.checker.check_model(loaded,full_check=True); checker=True
    inferred=onnx.shape_inference.infer_shapes(loaded); inferred_ok=True
    for value in list(inferred.graph.value_info)+list(inferred.graph.output):
        dims=[d.dim_value if d.HasField("dim_value") else d.dim_param for d in value.type.tensor_type.shape.dim]
        inferred_shapes.append({"name":value.name,"dims":dims})
    if ort_available:
        import onnxruntime as ort
        session=ort.InferenceSession(str(onnx_path),providers=["CPUExecutionProvider"]); got=session.run(None,{"x":x.cpu().numpy(),"context":context.cpu().numpy()})[0]; ort_error=float(abs(got-ref).max()); ort_ok=True
except Exception as exc: message=f"{type(exc).__name__}: {str(exc).splitlines()[0]}"
metrics={"onnx_available":onnx_available,"onnxruntime_available":ort_available,"export_succeeded":exported,"checker_passed":checker,"shape_inference_passed":inferred_ok,"ort_executed":ort_ok,"ort_max_error":ort_error,"onnx_bytes":onnx_path.stat().st_size if onnx_path.exists() else 0,"inferred_shapes":inferred_shapes,"initializer_shapes":{"proj.weight":[7,12],"gate.weight":[7,5],"out.weight":[3,7]},"message":message}
analysis=(f"ONNX export/checker/shape-inference gates were {exported}/{checker}/{inferred_ok}; ONNX Runtime executed={ort_ok} "
          f"with max error {ort_error:.3e}. The graph occupied {metrics['onnx_bytes']:,} bytes and retained the physical width 7 "
          "through both input projections and the output consumer.")


/tmp/ipykernel_457393/2184966518.py:10: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,(x,context),onnx_path,input_names=["x","context"],output_names=["logits"],dynamic_axes={"x":{0:"batch"},"context":{0:"batch"},"logits":{0:"batch"}},opset_version=17,dynamo=False)


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| ONNX available | yes |
| Export succeeded | yes |
| Checker passed | yes |
| Shape inference passed | yes |
| ORT executed | yes |
| ORT max error | 0.000000 |
| ONNX bytes | 1,722 bytes |


## 7. Interpret rather than merely print

ONNX export/checker/shape-inference gates were True/True/True; ONNX Runtime executed=True with max error 5.960e-08. The graph occupied 1,722 bytes and retained the physical width 7 through both input projections and the output consumer.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`native-backend`**. A named non-PyTorch backend executed and its checker/runtime output is retained. This still does not transfer to another backend or workload.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 19,
    "title": 'ONNX Export, Graph Repair, and Shape Consistency',
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'A pruned ONNX graph is deliverable only after structural checks and independent runtime parity confirm the new shape contract.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 19,
  "title": "ONNX Export, Graph Repair, and Shape Consistency",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260827
  },
  "evidence_label": "native-backend",
  "metrics": {
    "onnx_available": true,
    "onnxruntime_available": true,
    "export_succeeded": true,
    "checker_passed": true,
    "shape_inference_passed": true,
    "ort_executed": true,
    "ort_max_error": 5.960464477539063e-08,
    "onnx_bytes": 1722,
    "inferred_shapes": [
      {
        "name": "/proj/Gemm_output_0",
        "dims": [
          "batch",
          7
        ]
      },
      {
        "name": "/gate/MatMul_output_0",
        "dims": [
          "batch",
          7
        ]
      },
      {
        "name": "/Add_output_0",
        "dims": [
          "batch",
          7
        ]
      },
      {
        "name": "/Constant_output_0",
        "d

## 9. Make the bounded decision

> A pruned ONNX graph is deliverable only after structural checks and independent runtime parity confirm the new shape contract.

**Acceptance/rollback:** Accept the exported graph only when checker, shape inference audit, initializer dimensions, and target-runtime parity all pass.

**Failure analysis:** Tracer warnings and constant folding can hide data-dependent behavior. Shape inference may remain partial, and ONNX Runtime success does not guarantee TensorRT support. Multi-profile production shapes must be tested separately.


## 10. Extend the evidence

Add dynamic batch and sequence axes, deliberately corrupt one initializer to verify the audit fails, then test the repaired graph in the final deployment runtime.

The full evidence boundary and references are in [`README.md`](README.md).
